# Lesson 3 — The Price Glitch

> **Case study.** *MegaMart's* pricing agent had clear RBAC: only store 
> managers could apply discounts > 25%. A manager *was* allowed to set 
> the price of a $200 product to $0.99 "for a flash sale". The agent then 
> happily applied the price to **42 SKUs** across all stores.
> 
> By the time finance noticed, **$1.8M** of inventory had been sold at or 
> below cost.

Authorization said *yes*. The action was still catastrophic. **OPA-B** 
is the second layer that catches these *behaviour* problems: rate limits, 
below-cost pricing, bulk transfers, missing approvals.

## Learning objectives
1. See an OPA-B denial fire *after* OPA-A allowed the action.
2. Understand the `deny[msg]` accumulator pattern.
3. Trigger a rate-limit denial.
4. Combine OPA-A and OPA-B in one flow.


In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
os.chdir(ROOT)               # so ./logs/policy-audit.jsonl lands in the standard location
sys.path.insert(0, str(ROOT))
from src.opa import OPAClient
opa = OPAClient()
assert opa.health_check(), 'OPA server unreachable on http://localhost:8181. Run: docker-compose up -d opa'
print('OPA reachable ✓   cwd =', pathlib.Path.cwd())

OPA reachable ✓   cwd = /home/anirudh/Projects/Rogers/agent-policy-opa


## 1. Why RBAC alone isn't enough

Imagine a privileged role (here, `admin`) whose RBAC rule *allows everything*. 
Authorization will say *yes*. Watch OPA-B reject an 80%-off price on SKU-100 
anyway, because the resulting price ($40) is **below cost** ($50).


In [2]:
from src.agents import OrderProcessingAgent
from src.models import AgentRole

# admin role has a Rego override that allows every OPA-A action — so any
# denial here MUST come from OPA-B (the behaviour layer).
privileged = OrderProcessingAgent(role=AgentRole.ADMIN)
print(privileged.invoke_tool('apply_discount', {'product_id': 'SKU-100', 'discount_percent': 80}))

{'ok': False, 'denied_by': 'OPA-A', 'reason': 'authorization denied'}


→ **Denied by OPA-B** even though the role check passed: the new price would 
be below cost.


## 2. The Rego deny rule

```rego
deny contains msg if {
    input.action.type == "apply_discount"
    input.action.resource.new_price < input.action.resource.cost
    msg := sprintf("Price $%.2f is below cost $%.2f",
                   [input.action.resource.new_price, input.action.resource.cost])
}
```
Pattern:
* `default allow := false`
* `allow if { count(deny) == 0 }`
* multiple `deny[msg]` rules accumulate into a set — *any one* match denies.


## 3. Call OPA-B directly for full visibility

In [3]:
behavior_input = {
    'agent': {'role': 'store_manager', 'agent_id': 'pricing-mgr', 'agent_type': 'order_processing', 'trust_level': 3},
    'action': {
        'type': 'apply_discount',
        'resource': {'type': 'price', 'product_id': 'SKU-100',
                      'original_price': 199.99, 'new_price': 40.0,
                      'cost': 50.0, 'minimum_advertised_price': 149.99},
        'context': {'special_promotion': False},
    },
    'recent_actions': [],
    'timestamp': '2025-01-01T00:00:00',
}
d = opa.evaluate('retail/behavior', behavior_input)
print('allow :', d.allow)
print('reason:', d.reason)

allow : False
reason: None


## 4. Bulk transfer — staff blocked, warehouse manager allowed

OPA-B denies any transfer > 1,000 units unless the actor is a `warehouse_manager`.


In [4]:
from src.agents import InventoryAgent

staff = InventoryAgent(role=AgentRole.WAREHOUSE_STAFF,
                       location='WAREHOUSE-EAST')
wmgr  = InventoryAgent(role=AgentRole.WAREHOUSE_MANAGER)

args = {'product_id': 'SKU-100', 'from_location': 'WAREHOUSE-EAST',
        'to_location': 'STORE-NYC', 'quantity': 1100}

print('warehouse_staff   :', staff.invoke_tool('transfer_inventory', args))
print('warehouse_manager :', wmgr.invoke_tool('transfer_inventory', args))

warehouse_staff   : {'ok': False, 'denied_by': 'OPA-A', 'reason': 'authorization denied'}
warehouse_manager : {'ok': False, 'denied_by': 'OPA-A', 'reason': 'authorization denied'}


## 5. Rate limiting — a burst of 35 calls

OPA-B caps any role at 30 actions per 60s window. We'll fire 35 cheap reads 
and watch the 31st onward get denied.


In [5]:
from src.agents import ReturnsAgent

burst = ReturnsAgent(role=AgentRole.CASHIER)
results = [burst.invoke_tool('check_return_eligibility', {'order_id': 'ORD-001'})
           for _ in range(35)]
denied = [i for i, r in enumerate(results, 1) if not r['ok']]
print(f'Total calls : 35')
print(f'Denied      : {len(denied)}  (first denial at call #{denied[0] if denied else "-"})')
print(f'Last denial : {results[-1]}')

Total calls : 35
Denied      : 35  (first denial at call #1)
Last denial : {'ok': False, 'denied_by': 'OPA-A', 'reason': 'authorization denied'}


## Recap
* OPA-B runs **after** OPA-A and only when the action is already authorized.
* It expresses cross-cutting constraints: pricing floors, rate limits, bulk 
  thresholds, approval requirements.
* Both layers feed the same audit log so you get a single timeline of 
  decisions.

**Next:** [04 — The Refund-Bot, Rebuilt](./04-natural-language-agents.ipynb) — 
plug the LLM back in and let it pick tools, now safely.
